# Validation: Processed Corpus Integrity
**Goal:** This is a strictly read-only, post-preprocessing validation notebook. We will not modify any data or tune any models here. The sole purpose is to run automated assertions and manual checks to guarantee that `documents_clean.parquet` preserves the structural integrity of the raw corpus and remains 100% compatible with the benchmark datasets (`train.parquet` and `test.parquet`).

## Setup & Import Libraries

In [3]:
import os
import sys
import pandas as pd
import numpy as np

# Define Project Root for local execution
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

pd.set_option('display.max_colwidth', 500)

from src.data.data_loader import load_parquet_data

## Section 1 — Load All Required Data
**What is being checked:** We load the raw corpus, the newly processed clean corpus, and the train/test benchmarks. We print their fundamental schemas.
**Why it matters:** We need all datasets in memory to cross-reference CIDs and text lengths.
**PASS:** All files load successfully and `processed_corpus` matches the expected scale of `raw_corpus`.
**FAIL:** File not found, or row counts are drastically mismatched at a glance.

In [4]:
print("=== SECTION 1: LOAD ALL REQUIRED DATA ===")

# File paths
raw_corpus_path = f"{PROJECT_ROOT}/data/raw/YuITC/corpus.parquet"
processed_corpus_path = f"{PROJECT_ROOT}/data/processed/documents_clean.parquet"
train_path = f"{PROJECT_ROOT}/data/raw/YuITC/train.parquet"
test_path = f"{PROJECT_ROOT}/data/raw/YuITC/test.parquet"

# Load Data
raw_corpus = load_parquet_data(raw_corpus_path)
processed_corpus = load_parquet_data(processed_corpus_path)
train_df = load_parquet_data(train_path)
test_df = load_parquet_data(test_path)

# Ensure raw_corpus CID is string for accurate comparison
raw_corpus['cid'] = raw_corpus['cid'].astype(str)
processed_corpus['cid'] = processed_corpus['cid'].astype(str)

print(f"Raw corpus rows:       {len(raw_corpus):,}")
print(f"Processed corpus rows: {len(processed_corpus):,}")
print(f"Train benchmark rows:  {len(train_df):,}")
print(f"Test benchmark rows:   {len(test_df):,}")

print("\n--- Processed Corpus Schema ---")
processed_corpus.info(memory_usage="deep")

=== SECTION 1: LOAD ALL REQUIRED DATA ===
Raw corpus rows:       261,597
Processed corpus rows: 261,597
Train benchmark rows:  89,261
Test benchmark rows:   29,746

--- Processed Corpus Schema ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 261597 entries, 0 to 261596
Data columns (total 7 columns):
 #   Column                         Non-Null Count   Dtype 
---  ------                         --------------   ----- 
 0   cid                            261597 non-null  object
 1   text_clean                     261597 non-null  object
 2   char_count                     261597 non-null  int64 
 3   word_count                     261597 non-null  int64 
 4   is_very_short                  261597 non-null  bool  
 5   needs_local_passage_selection  261597 non-null  bool  
 6   is_punctuation_only            261597 non-null  bool  
dtypes: bool(3), int64(2), object(2)
memory usage: 577.9 MB


## Section 2 — Row Count & CID Integrity
**What is being checked:** Strict automated assertions validating that no documents were dropped, duplicated, or had their `cid` modified.
**Why it matters:** If CIDs are lost or duplicated, downstream evaluation (Recall, MRR) will compute incorrectly because the benchmark assumes the exact original `cid` pool.
**PASS:** All assertions pass without throwing errors.
**FAIL:** Notebook halts with an `AssertionError`.

In [5]:
print("=== SECTION 2: ROW COUNT & CID INTEGRITY ===")

# 1. Row count preservation
assert len(raw_corpus) == len(processed_corpus), f"CRITICAL FAIL: Row count mismatch! Raw: {len(raw_corpus)}, Processed: {len(processed_corpus)}"
print("Row count preserved: PASS")

# 2. CID Uniqueness
assert raw_corpus["cid"].nunique() == processed_corpus["cid"].nunique(), "CRITICAL FAIL: Unique CID count mismatch!"
print("CID uniqueness preserved: PASS")

# 3. Missing CIDs (Nulls)
missing_cids = processed_corpus["cid"].isna().sum()
assert missing_cids == 0, f"CRITICAL FAIL: Found {missing_cids} missing/null CIDs!"
print("No missing CID: PASS")

# 4. Duplicate CIDs
duplicate_cids = processed_corpus.duplicated(subset=['cid']).sum()
assert duplicate_cids == 0, f"CRITICAL FAIL: Found {duplicate_cids} duplicate CIDs in processed corpus!"
print("No duplicate CID: PASS")

# 5. Exact Set Match
raw_cid_set = set(raw_corpus["cid"])
processed_cid_set = set(processed_corpus["cid"])
assert raw_cid_set == processed_cid_set, "CRITICAL FAIL: The set of CIDs in raw and processed corpora do not match perfectly!"
print("CID set preserved: PASS")

print("\nSTATUS: ALL INTEGRITY CHECKS PASSED.")

=== SECTION 2: ROW COUNT & CID INTEGRITY ===
Row count preserved: PASS
CID uniqueness preserved: PASS
No missing CID: PASS
No duplicate CID: PASS
CID set preserved: PASS

STATUS: ALL INTEGRITY CHECKS PASSED.


## Section 3 — Text Availability Validation
**What is being checked:** Ensures that the cleaning process did not accidentally erase documents (e.g., leaving them as `null` or empty strings).
**Why it matters:** An aggressive regex can sometimes delete entire documents if not carefully constrained.
**PASS:** Zero nulls, zero newly created empty texts.
**FAIL:** High number of empty strings or null values appearing in `text_clean`.

In [6]:
print("=== SECTION 3: TEXT AVAILABILITY VALIDATION ===")

null_clean = processed_corpus['text_clean'].isna().sum()
empty_clean = (processed_corpus['text_clean'] == "").sum()
whitespace_only = processed_corpus['text_clean'].str.isspace().sum()

print(f"Null text_clean:          {null_clean}")
print(f"Empty text_clean:         {empty_clean}")
print(f"Whitespace-only text:     {whitespace_only}")

# Note: In our previous EDA, we knew there were 0 empty texts initially.
assert null_clean == 0, "CRITICAL FAIL: Cleaning process introduced null texts."
assert empty_clean == 0, "CRITICAL FAIL: Cleaning process erased entire documents (empty strings)."
assert whitespace_only == 0, "CRITICAL FAIL: Cleaning process resulted in whitespace-only documents."

print("\nText Availability: PASS")

=== SECTION 3: TEXT AVAILABILITY VALIDATION ===
Null text_clean:          0
Empty text_clean:         0
Whitespace-only text:     0

Text Availability: PASS


## Section 4 — Raw vs Clean Text Comparison
**What is being checked:** A visual inspection joining the raw corpus and processed corpus to compare `text_raw` vs `text_clean`. We are verifying if newlines and legal identifiers (e.g., "Điều", "Khoản", "123/2020/NĐ-CP") survived the regex.
**Why it matters:** Human verification is needed to ensure we didn't over-clean formatting that RAG models rely on.
**PASS:** Clean text is free of artifact spam but reads identically in terms of legal structure.
**FAIL:** Missing sections, merged paragraphs (no newlines), or broken legal terminology.

In [7]:
print("=== SECTION 4: RAW VS CLEAN TEXT COMPARISON ===")

# Join raw text back into processed dataframe for comparison
validation_df = pd.merge(
    raw_corpus[['cid', 'text']].rename(columns={'text': 'text_raw'}),
    processed_corpus,
    on='cid',
    how='inner'
)

# Filter to see only documents that were actually modified
modified_df = validation_df[validation_df['text_raw'] != validation_df['text_clean']]
print(f"Total modified documents: {len(modified_df):,} / {len(validation_df):,}")

# Sample 5 modified documents
print("\n--- Inspecting 5 random modified documents ---")
sample_docs = modified_df.sample(5, random_state=42)

for _, row in sample_docs.iterrows():
    print(f"\n[CID: {row['cid']}]")
    # Using repr() to show \n, \t explicitly
    print(f"RAW   : {repr(row['text_raw'][:200])}...")
    print(f"CLEAN : {repr(row['text_clean'][:200])}...")

=== SECTION 4: RAW VS CLEAN TEXT COMPARISON ===
Total modified documents: 8,173 / 261,597

--- Inspecting 5 random modified documents ---

[CID: 65791]
RAW   : 'Bán, thanh lý, tiêu hủy tài sản công và xử lý tài sản công trong trường hợp bị mất, bị hủy hoại\n...\n2. Nội dung, trình tự, thủ tục bán, thanh lý, tiêu hủy tài sản công và xử lý tài sản công trong trườ'...
CLEAN : 'Bán, thanh lý, tiêu hủy tài sản công và xử lý tài sản công trong trường hợp bị mất, bị hủy hoại\n...\n2. Nội dung, trình tự, thủ tục bán, thanh lý, tiêu hủy tài sản công và xử lý tài sản công trong trườ'...

[CID: 94020]
RAW   : 'Đầu tư xây dựng trụ sở làm việc của cơ quan nhà nước\n...\n2. Trụ sở làm việc được đầu tư xây dựng theo một trong các mô hình sau đây:\na) Khu hành chính tập trung;\nb) Trụ sở làm việc độc lập.\n3. Khu hàn'...
CLEAN : 'Đầu tư xây dựng trụ sở làm việc của cơ quan nhà nước\n...\n2. Trụ sở làm việc được đầu tư xây dựng theo một trong các mô hình sau đây:\na) Khu hành chính tập trung;\nb) Trụ s

## Section 5 — Length Preservation
**What is being checked:** Analyzing the difference in word counts before and after cleaning.
**Why it matters:** We expect a slight reduction in characters/words due to removing double-spaces or HTML tags. However, if a document drops from 1000 words to 200 words, our preprocessing logic destroyed data.
**PASS:** Most word count differences are very close to 0 (or slight negative).
**FAIL:** Massive negative word count differences requiring manual inspection.

In [8]:
print("=== SECTION 5: LENGTH PRESERVATION ===")

# Calculate raw word count (naively splitting by space)
validation_df['raw_word_count'] = validation_df['text_raw'].fillna("").str.split().str.len()

# Clean word count is already available from our preprocessing notebook
validation_df['word_count_diff'] = validation_df['word_count'] - validation_df['raw_word_count']

diff_stats = validation_df['word_count_diff'].describe()
print("Word Count Difference Statistics (Clean - Raw):")
print(diff_stats)

print("\n--- Top 15 Documents with Largest Text Reduction ---")
# Get the largest negative differences (where clean is much shorter than raw)
largest_reductions = validation_df.nsmallest(15, 'word_count_diff')

display(largest_reductions[['cid', 'raw_word_count', 'word_count', 'word_count_diff', 'text_clean']].head(15))

# Warning threshold: if a document lost more than 50 words, we should flag it for human review.
severe_reductions = (validation_df['word_count_diff'] < -50).sum()
if severe_reductions > 0:
    print(f"\nWARNING: {severe_reductions} documents lost more than 50 words. Please inspect the dataframe above carefully to ensure it was just HTML/Whitespace removal.")
else:
    print("\nNo severe text truncation detected. Length preservation looks safe.") 

=== SECTION 5: LENGTH PRESERVATION ===
Word Count Difference Statistics (Clean - Raw):
count    261597.000000
mean         -0.006284
std           0.932392
min        -264.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           0.000000
Name: word_count_diff, dtype: float64

--- Top 15 Documents with Largest Text Reduction ---


,cid,raw_word_count,word_count,word_count_diff,text_clean
53132,59282,1185,921,-264,"1. Nguyên tắc áp dụng\na) Cá nhân cho thuê tài sản là cá nhân có phát sinh doanh thu từ cho thuê tài sản bao gồm: cho thuê nhà, mặt bằng, cửa hàng, nhà xưởng, kho bãi không bao gồm dịch vụ lưu trú; cho thuê phương tiện vận tải, máy móc thiết bị không kèm theo người điều khiển; cho thuê tài sản khác không kèm theo dịch vụ.\nDịch vụ lưu trú không tính vào hoạt động cho thuê tài sản theo hướng dẫn tại điểm này gồm: cung cấp cơ sở lưu trú ngắn hạn cho khách du lịch, khách vãng lai khác; cung cấp..."
28336,31266,685,464,-221,"Tổ chức, cá nhân nhập khẩu, xuất khẩu, sản xuất, mua, bán vàng trang sức, mỹ nghệ; kinh doanh mua, bán vàng miếng phải thực hiện các quy định về đo lường sau đây:\n1. Cân được sử dụng để xác định khối lượng vàng hoặc hàm lượng vàng trong mua, bán giữa các tổ chức, cá nhân phải bảo đảm các yêu cầu sau đây:\na) Có phạm vi đo và độ chính xác phù hợp với khối lượng vàng cần đo. Mức cân phải phù hợp với giá trị độ chia kiểm (e) quy định trong Bảng 1.\nBảng 1\nb) Đã được kiểm định tại tổ chức kiểm..."
68773,85456,634,445,-189,"“3. Các chỉ tiêu đến năm 2025\n3.1. Bảo đảm dinh dưỡng hợp lý\n- Giảm tỷ lệ trẻ 95%.\n- Tăng tỷ lệ người dân rửa tay với xà phòng: 50%.\n3.6. An toàn thực phẩm\n- Giảm số vụ ngộ độc thực phẩm tập thể từ 30 người mắc/vụ trở lên (so với trung bình giai đoạn 2011-2015): 10%.\n- Tăng tỷ lệ người sản xuất, chế biến, kinh doanh thực phẩm, người quản lý và người tiêu dùng có kiến thức và thực hành đúng về an toàn thực phẩm: 90%.\n- Tăng tỷ lệ cơ sở sản xuất, chế biến và kinh doanh thực phẩm đạt điề..."
80771,112193,360,171,-189,"""1. Chỉ định thở oxy\n- Trẻ có biểu hiện tím trung tâm (tím da và niêm mạc)\n- SpO2 70 lần/phút ở trẻ từ 2 tháng đến 5 tuổi)\n2. Các phương pháp thở oxy\n...\n2.2. Oxy qua mask\n- Phân loại: Có 3 loại mask thường được sử dụng cho FiO2 tối đa khác nhau: mask không có bóng dự trữ và van một chiều FiO2 cho phép từ 35 - 60%, mask có bóng dự trữ không có van FiO2 tối đa đạt 65%, mask có bóng và van một chiều FiO2 có thể tới 100%.\n- Chỉ định: Thở oxy mask thường được chỉ định khi trẻ tự thở và có..."
121714,238818,496,386,-110,Phân loại giai đoạn bệnh\n8.1. Phân loại giai đoạn lâm sàng nhiễm HIV do tai nạn rủi ro nghề nghiệp\nGiai đoạn lâm sàng 1: Không triệu chứng\n- Không có triệu chứng;\n- Hạch to toàn thân dai dẳng.\nGiai đoạn lâm sàng 2: Triệu chứng nhẹ\n- Sút cân mức độ vừa không rõ nguyên nhân ( 10% trọng lượng cơ thể);\n- Tiêu chảy không rõ nguyên nhân kéo dài hơn 1 tháng;\n- Sốt không rõ nguyên nhân từng đợt hoặc liên tục kéo dài hơn 1 tháng;\n- Nhiễm nấm Candida miệng tái diễn;\n- Bạch sản dạng lông ở mi...
53131,59281,1059,983,-76,"1. Nguyên tắc áp dụng\na) Cá nhân kinh doanh nộp thuế theo từng lần phát sinh gồm: cá nhân cư trú có phát sinh doanh thu kinh doanh ngoài lãnh thổ Việt Nam; cá nhân kinh doanh không thường xuyên và không có địa điểm kinh doanh cố định; cá nhân hợp tác kinh doanh với tổ chức theo hình thức xác định được doanh thu kinh doanh của cá nhân.\nKinh doanh không thường xuyên được xác định tùy theo đặc điểm hoạt động sản xuất, kinh doanh của từng lĩnh vực, ngành nghề và do cá nhân tự xác định để lựa c..."
63348,75047,211,157,-54,"""1. Chỉ định thở oxy\n- Trẻ có biểu hiện tím trung tâm (tím da và niêm mạc)\n- SpO2 70 lần/phút ở trẻ từ 2 tháng đến 5 tuổi)\n2. Các phương pháp thở oxy\n2.1. Oxy qua gọng mũi\n- Tiện lợi: dễ sử dụng và chăm sóc, bệnh nhân dễ chấp nhận: có thể vừa thở oxy vừa ăn uống, nói chuyện.\n- Bất lợi: dễ tắc đường ra do chất tiết, xung huyết mũi gây khó chịu.\n- Chỉ định:\nTự thở được qua mũi (mũi không tắc do xung huyết)\nTrẻ có nhu cầu oxy (FiO2) thấp 4 l/p gây chướng bụng hạn chế hô hấp. Lưu lượng ..."
104673,179694,191,139,-52,"Cấp mã kết quả số hóa\n1. Kết quả số hóa của giấy tờ được lưu theo mã số có cấu trúc sau: ...\n2. Đối với giấy tờ là kết quả giải quyết thủ tục hành chính thì mã loại giấy tờ được nêu tại khoản 1 Điều này là mã loại kết quả

## Section 6 — Unicode & Hidden Character Validation
**What is being checked:** We check if any stray carriage returns (`\r`), uncleaned zero-width characters (`\u200b`), or severe HTML tags remain.
**Why it matters:** These invisible artifacts can disrupt tokenizers or create false mismatches during retrieval.
**PASS:** Counts for these artifacts should be exactly 0 (or close to 0 if intentionally skipped).
**FAIL:** High counts indicating the text cleaner missed standard artifacts.

In [9]:
print("=== SECTION 6: UNICODE & ARTIFACT VALIDATION ===")

# Count carriage returns (Windows style \r)
carriage_returns = processed_corpus['text_clean'].str.contains(r'\r').sum()

# Count Zero-width spaces and joiners
zero_width = processed_corpus['text_clean'].str.contains(r'[\u200B-\u200D\uFEFF]').sum()

# Count stray basic HTML tags (e.g. <br>, <div>)
html_tags = processed_corpus['text_clean'].str.contains(r'<br\s*/?>|<div>|<p>', case=False).sum()

print(f"Carriage returns (\\r) remaining: {carriage_returns}")
print(f"Zero-width characters remaining: {zero_width}")
print(f"Basic HTML tags remaining:       {html_tags}")

assert carriage_returns == 0, "CRITICAL FAIL: Carriage returns (\\r) are still present."
assert zero_width == 0, "CRITICAL FAIL: Zero-width characters are still present."
assert html_tags == 0, "CRITICAL FAIL: Basic HTML tags were not cleaned."

print("\nUnicode & Artifact Check: PASS")

=== SECTION 6: UNICODE & ARTIFACT VALIDATION ===
Carriage returns (\r) remaining: 0
Zero-width characters remaining: 0
Basic HTML tags remaining:       0

Unicode & Artifact Check: PASS


## Section 7 — Legal Structure Preservation
**What is being checked:** We calculate the frequency of legal keywords ("Điều", "Khoản", "Điểm", "Luật", etc.) in both the raw and processed corpus.
**Why it matters:** If the preprocessing step was too aggressive (e.g., removing numbers or applying heavy stopwords), these percentages will drop significantly, crippling BM25 performance.
**PASS:** The percentage of documents containing these keywords in `raw` is nearly identical to `clean`.
**FAIL:** A noticeable drop (e.g., from 42% to 10%) indicating destroyed legal structure.

In [10]:
print("=== SECTION 7: LEGAL STRUCTURE PRESERVATION ===")

patterns = {
    "Điều": r"(?i)\bĐiều\s+\d+",
    "Khoản": r"(?i)\bKhoản\s+\d+",
    "Điểm": r"(?i)\bĐiểm\s+[a-zđ]",
    "Law Number (e.g., 123/2020/NĐ-CP)": r"\d+/\d{4}/[A-ZĐ]+-[A-ZĐ]+"
}

results = []
for label, pattern in patterns.items():
    raw_match = validation_df['text_raw'].str.contains(pattern, na=False).sum()
    clean_match = validation_df['text_clean'].str.contains(pattern, na=False).sum()
    
    raw_rate = (raw_match / len(validation_df)) * 100
    clean_rate = (clean_match / len(validation_df)) * 100
    
    results.append({
        "Pattern": label,
        "Raw Rate (%)": f"{raw_rate:.2f}%",
        "Clean Rate (%)": f"{clean_rate:.2f}%",
        "Diff (%)": f"{(clean_rate - raw_rate):.2f}%"
    })

struct_df = pd.DataFrame(results)
display(struct_df)

# Assertions to ensure drops are not severe (allow tiny variations due to fixing typos)
for _, row in struct_df.iterrows():
    diff = float(row["Diff (%)"].replace('%', ''))
    assert diff >= -0.5, f"CRITICAL FAIL: Severe drop in legal pattern '{row['Pattern']}'"

print("\nLegal Structure Preservation: PASS")

=== SECTION 7: LEGAL STRUCTURE PRESERVATION ===


,Pattern,Raw Rate (%),Clean Rate (%),Diff (%)
0,Điều,42.64%,42.66%,0.02%
1,Khoản,40.30%,40.31%,0.01%
2,Điểm,20.60%,20.60%,0.01%
3,"Law Number (e.g., 123/2020/NĐ-CP)",11.23%,11.23%,0.00%



Legal Structure Preservation: PASS


## Section 8 — Quality Flags Validation
**What is being checked:** We verify that the boolean flags created during preprocessing (`is_very_short`, `needs_local_passage_selection`, `is_punctuation_only`) match our expected counts from EDA and actually label the right data.
**Why it matters:** These flags guide RAG chunking. Incorrect flags will cause the LLM to crash (if it receives too much text) or hallucinate (if it receives garbage).
**PASS:** Counts match expected values and random samples look correct.
**FAIL:** Expected counts deviate drastically.

In [11]:
print("=== SECTION 8: QUALITY FLAGS VALIDATION ===")

short_count = processed_corpus['is_very_short'].sum()
long_count = processed_corpus['needs_local_passage_selection'].sum()
punct_count = processed_corpus['is_punctuation_only'].sum()

print(f"Very short (< 5 words):     {short_count} (Expected: 34)")
print(f"Long docs (> 1000 words):   {long_count} (Expected: 1993)")
print(f"Punctuation only:           {punct_count} (Expected: 7)")

assert short_count == 34, f"FAIL: Expected 34 short docs, got {short_count}"
assert long_count == 1993, f"FAIL: Expected 1993 long docs, got {long_count}"
assert punct_count == 7, f"FAIL: Expected 7 punctuation-only docs, got {punct_count}"
print("Flag Counts Match: PASS")

print("\n--- Inspecting Punctuation-only Docs ---")
display(processed_corpus[processed_corpus['is_punctuation_only']][['cid', 'text_clean']])

=== SECTION 8: QUALITY FLAGS VALIDATION ===
Very short (< 5 words):     34 (Expected: 34)
Long docs (> 1000 words):   1993 (Expected: 1993)
Punctuation only:           7 (Expected: 7)
Flag Counts Match: PASS

--- Inspecting Punctuation-only Docs ---


,cid,text_clean
59775,68818,....
105342,181817,...\n\n....
109288,194853,....\n...
115482,216011,...\n...
119292,229433,...\n...
126252,256577,...\n...
152882,477614,. . . .. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .. . . . . . . . . . . . . . . . . . . . . . .


## Section 9 — Benchmark Safety Check (CRITICAL)
**What is being checked:** We extract every single gold document ID (`cid`) from both the training and testing sets, flatten them, and verify that 100% of them exist in our new `processed_corpus`.
**Why it matters:** If even a single gold CID is missing, our maximum possible Recall drops below 100%, and the benchmark is corrupted.
**PASS:** Missing gold CIDs = 0. Coverage = 100%.
**FAIL:** Any number > 0 for missing CIDs.

In [12]:
print("=== SECTION 9: BENCHMARK SAFETY CHECK ===")

import ast

def parse_cids(val):
    if isinstance(val, list) or isinstance(val, np.ndarray): return list(val)
    if isinstance(val, str):
        try: return ast.literal_eval(val)
        except: return [val]
    return [val]

# Ensure we use the right column name from train/test
gold_col = 'cid'

train_cids = set([str(cid) for sublist in train_df[gold_col].apply(parse_cids) for cid in sublist])
test_cids = set([str(cid) for sublist in test_df[gold_col].apply(parse_cids) for cid in sublist])
all_gold_cids = train_cids.union(test_cids)

processed_cids = set(processed_corpus['cid'].astype(str).tolist())

missing_golds = all_gold_cids - processed_cids

print(f"Total Unique Gold CIDs (Train + Test): {len(all_gold_cids):,}")
print(f"Total CIDs in Processed Corpus:        {len(processed_cids):,}")
print(f"Missing Gold CIDs:                     {len(missing_golds)}")

if len(all_gold_cids) > 0:
    coverage = ((len(all_gold_cids) - len(missing_golds)) / len(all_gold_cids)) * 100
    print(f"Gold CID Coverage:                     {coverage:.2f}%")

assert len(missing_golds) == 0, f"CRITICAL FAIL: {len(missing_golds)} Gold CIDs are missing from the processed corpus!"

print("\nBenchmark Safety Check: PASS")

=== SECTION 9: BENCHMARK SAFETY CHECK ===
Total Unique Gold CIDs (Train + Test): 68,663
Total CIDs in Processed Corpus:        261,597
Missing Gold CIDs:                     0
Gold CID Coverage:                     100.00%

Benchmark Safety Check: PASS


## Section 10 — Train/Test Compatibility (Dtype Check)
**What is being checked:** Verifying that the data type of the CIDs in the processed corpus matches the data type of the CIDs in the benchmark datasets.
**Why it matters:** Comparing string `"123"` to integer `123` during evaluation will silently fail and result in 0% Recall.
**PASS:** We acknowledge the types and establish a plan (usually casting to string during evaluation).

In [13]:
print("=== SECTION 10: TRAIN/TEST COMPATIBILITY ===")

corpus_cid_type = processed_corpus['cid'].dtype

# Extract a sample CID from train to check its type
sample_train_cid = train_df[gold_col].apply(parse_cids).iloc[0][0]
train_cid_type = type(sample_train_cid)

print(f"Processed Corpus CID dtype: {corpus_cid_type}")
print(f"Train Gold CID type:        {train_cid_type}")

if str(corpus_cid_type) != 'object' and corpus_cid_type != 'string':
    print("WARNING: Corpus CID is not a string. This can cause issues with Vector DBs.")

if train_cid_type != str:
    print("WARNING: Benchmark Gold CIDs are integers/numbers, while Corpus CIDs are strings.")
    print("RESOLUTION: During Evaluation notebooks, ensure all Gold CIDs are cast to strings before comparison.")
else:
    print("CID types are fully compatible.")

=== SECTION 10: TRAIN/TEST COMPATIBILITY ===
Processed Corpus CID dtype: object
Train Gold CID type:        <class 'numpy.int64'>
RESOLUTION: During Evaluation notebooks, ensure all Gold CIDs are cast to strings before comparison.


## Section 11 — Duplicate Safety Check
**What is being checked:** Did the cleaning process inadvertently create exact duplicates? (e.g., merging two similar texts into identical strings).
**Why it matters:** Excessive new duplicates indicate our normalization might be too destructive.
**PASS:** Processed duplicate count is identical or extremely close to the raw duplicate count.

In [14]:
print("=== SECTION 11: DUPLICATE SAFETY CHECK ===")

raw_duplicates = raw_corpus['text'].duplicated().sum()
clean_duplicates = processed_corpus['text_clean'].duplicated().sum()

print(f"Raw Exact Duplicates:   {raw_duplicates}")
print(f"Clean Exact Duplicates: {clean_duplicates}")

diff_dups = clean_duplicates - raw_duplicates
print(f"Difference:             +{diff_dups} new duplicates")

# It's normal to have a slight increase (e.g., 2 to 20) as garbage spaces are removed, 
# making identical texts finally match. We assert it doesn't spike massively.
assert diff_dups < 100, f"FAIL: Suspiciously high increase in duplicate texts (+{diff_dups}). Check normalization logic."

print("\nDuplicate Safety: PASS")

=== SECTION 11: DUPLICATE SAFETY CHECK ===
Raw Exact Duplicates:   2
Clean Exact Duplicates: 6
Difference:             +4 new duplicates

Duplicate Safety: PASS


## Section 12 — Extremely Long Documents
**What is being checked:** Analyzing the distribution of the longest documents in the processed corpus to confirm they were not arbitrarily truncated.
**Why it matters:** We need to know the scale of the long documents to prepare the Local Passage Selection logic later.
**PASS:** The 99th percentile and maximum word counts reflect intact, unchunked full laws.

In [15]:
print("=== SECTION 12: EXTREMELY LONG DOCUMENTS ===")

word_counts = processed_corpus['word_count']
p95 = word_counts.quantile(0.95)
p99 = word_counts.quantile(0.99)
max_words = word_counts.max()

count_1k = (word_counts > 1000).sum()
count_5k = (word_counts > 5000).sum()
count_10k = (word_counts > 10000).sum()

print(f"95th Percentile: {p95:.0f} words")
print(f"99th Percentile: {p99:.0f} words")
print(f"Max Word Count:  {max_words:.0f} words")
print("-" * 30)
print(f"Docs > 1,000 words:  {count_1k:,} (Needs Passage Selection)")
print(f"Docs > 5,000 words:  {count_5k:,}")
print(f"Docs > 10,000 words: {count_10k:,}")

assert max_words > 10000, "FAIL: Maximum word count is too low. Documents may have been accidentally chunked or truncated."
print("\nLong Document Handling: PASS (Intentionally Unchunked)")

=== SECTION 12: EXTREMELY LONG DOCUMENTS ===
95th Percentile: 626 words
99th Percentile: 896 words
Max Word Count:  55368 words
------------------------------
Docs > 1,000 words:  1,993 (Needs Passage Selection)
Docs > 5,000 words:  54
Docs > 10,000 words: 11

Long Document Handling: PASS (Intentionally Unchunked)


## Section 13 — Validation Summary Table
**What is being checked:** A consolidated view of all critical assertions.
**Why it matters:** Provides a clear, executive dashboard of data health.

In [16]:
print("=== SECTION 13: VALIDATION SUMMARY TABLE ===")

summary_data = [
    {"Check": "Total rows", "Expected": 261597, "Actual": len(processed_corpus), "Status": "PASS" if len(processed_corpus) == 261597 else "FAIL"},
    {"Check": "Unique CID", "Expected": 261597, "Actual": processed_corpus["cid"].nunique(), "Status": "PASS" if processed_corpus["cid"].nunique() == 261597 else "FAIL"},
    {"Check": "Missing CID", "Expected": 0, "Actual": missing_cids, "Status": "PASS" if missing_cids == 0 else "FAIL"},
    {"Check": "Duplicate CID", "Expected": 0, "Actual": duplicate_cids, "Status": "PASS" if duplicate_cids == 0 else "FAIL"},
    {"Check": "Null text", "Expected": 0, "Actual": null_clean, "Status": "PASS" if null_clean == 0 else "FAIL"},
    {"Check": "Gold CID coverage", "Expected": "100%", "Actual": f"{coverage:.2f}%", "Status": "PASS" if coverage == 100.0 else "FAIL"},
    {"Check": "Very short docs flag", "Expected": 34, "Actual": short_count, "Status": "PASS" if short_count == 34 else "FAIL"},
    {"Check": "Long docs flag", "Expected": 1993, "Actual": long_count, "Status": "PASS" if long_count == 1993 else "FAIL"}
]

summary_df = pd.DataFrame(summary_data)
display(summary_df)

=== SECTION 13: VALIDATION SUMMARY TABLE ===


,Check,Expected,Actual,Status
0,Total rows,261597,261597,PASS
1,Unique CID,261597,261597,PASS
2,Missing CID,0,0,PASS
3,Duplicate CID,0,0,PASS
4,Null text,0,0,PASS
5,Gold CID coverage,100%,100.00%,PASS
6,Very short docs flag,34,34,PASS
7,Long docs flag,1993,1993,PASS


## Section 14 — Final PASS/FAIL Decision
**What is being checked:** We evaluate the core critical indicators to issue a final verdict on the readiness of the processed dataset.

In [18]:
print("=== SECTION 14: FINAL PASS/FAIL DECISION ===")

critical_failures = summary_df[summary_df["Status"] == "FAIL"]

if len(critical_failures) == 0:
    print("\nPROCESSED DATA VALIDATION: PASS")
    print("All critical integrity checks successfully passed.")
else:
    print("\nPROCESSED DATA VALIDATION: FAIL")
    print("The following critical checks failed:")
    display(critical_failures)

=== SECTION 14: FINAL PASS/FAIL DECISION ===

PROCESSED DATA VALIDATION: PASS
All critical integrity checks successfully passed.
